In [22]:
# Setup paths and imports
from pathlib import Path
import sys, json

# Find project root by walking up until 'src' exists
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SRC = ROOT / 'src'
EDL = ROOT / 'edl' / 'finance_schema.edl'
ARTIFACTS = ROOT / 'artifacts'

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from edl_parser import parse_edl
from ddl_generator import generate_ddl, generate_fk_constraints
from schema_loader import load_schema_to_postgres
from generator_config import schema_to_generator_config
from synthetic_generator import generate_synthetic_data

print('Workspace:', ROOT)
print('Using EDL:', EDL)
assert EDL.exists(), 'EDL file not found: ' + str(EDL)

Workspace: C:\Users\Ankita\Documents\Quadyster\SyntheticDataHub_SDK
Using EDL: C:\Users\Ankita\Documents\Quadyster\SyntheticDataHub_SDK\edl\finance_schema.edl


## Parse EDL
Convert the EDL specification into an in-memory `SchemaMeta` model and inspect tables and relationships.

In [23]:
import importlib, edl_parser
importlib.reload(edl_parser)
schema = edl_parser.parse_edl(str(EDL))
print(f"Tables: {len(schema.tables)} | Relationships: {len(schema.relationships)}")
for tname, t in schema.tables.items():
    print('-', tname, '| fields:', len(t.fields), '| pk:', t.primary_key)
if schema.relationships:
    print('\nRelationships:')
    for r in schema.relationships:
        print(f"  {r.child_table}.{r.child_key} -> {r.parent_table}.{r.parent_key} ({r.cardinality})")

Tables: 8 | Relationships: 9
- Customer | fields: 7 | pk: customer_id
- Branch | fields: 4 | pk: branch_id
- Account | fields: 8 | pk: account_id
- Card | fields: 7 | pk: card_id
- Merchant | fields: 4 | pk: merchant_id
- Transaction | fields: 9 | pk: txn_id
- Loan | fields: 8 | pk: loan_id
- LoanPayment | fields: 5 | pk: payment_id

Relationships:
  Account.customer_id -> Customer.customer_id (1:N)
  Account.branch_id -> Branch.branch_id (1:N)
  Card.account_id -> Account.account_id (1:N)
  Transaction.account_id -> Account.account_id (1:N)
  Transaction.card_id -> Card.card_id (0:N)
  Transaction.merchant_id -> Merchant.merchant_id (0:N)
  Loan.customer_id -> Customer.customer_id (1:N)
  Loan.account_id -> Account.account_id (1:N)
  LoanPayment.loan_id -> Loan.loan_id (1:N)


## Generate DDL
Create `CREATE TABLE` statements and foreign key constraints. Preview and optionally save to file.

In [24]:
ddls = generate_ddl(schema)
fk_ddls = generate_fk_constraints(schema)

print('Sample DDLs:')
for i, (tname, sql) in enumerate(ddls.items()):
    print(f"\n-- {tname} --\n{sql}")
    if i >= 1:
        break

ddl_out = ARTIFACTS / 'ddl.sql'
with open(ddl_out, 'w', encoding='utf-8') as f:
    for sql in ddls.values():
        f.write(sql + '\n')
    for sql in fk_ddls.values():
        f.write(sql + '\n')
print('DDL written to', ddl_out)

Sample DDLs:

-- Customer --
CREATE TABLE IF NOT EXISTS "Customer" (
  "customer_id" VARCHAR(36) NOT NULL,
  "first_name" VARCHAR(50) NOT NULL,
  "last_name" VARCHAR(50) NOT NULL,
  "date_of_birth" DATE NOT NULL,
  "email" VARCHAR(100),
  "created_at" TIMESTAMP NOT NULL,
  "kyc_status" VARCHAR(20) NOT NULL,
  PRIMARY KEY ("customer_id")
);

-- Branch --
CREATE TABLE IF NOT EXISTS "Branch" (
  "branch_id" VARCHAR(16) NOT NULL,
  "branch_name" VARCHAR(100) NOT NULL,
  "city" VARCHAR(50) NOT NULL,
  "country" VARCHAR(50) NOT NULL,
  PRIMARY KEY ("branch_id")
);
DDL written to C:\Users\Ankita\Documents\Quadyster\SyntheticDataHub_SDK\artifacts\ddl.sql


## (Optional) Load Schema to Postgres
Start Postgres via Docker Compose before running the next cell:

```powershell
cd "$env:USERPROFILE\Documents\Quadyster\SyntheticDataHub_SDK"
docker compose up -d
```

Default connection: host=localhost, port=5432, db=finance_synth, user=postgres, password=postgres.

In [25]:
# Attempts to create tables and FKs in Postgres if available
try:
    load_schema_to_postgres(schema)
    print('Schema loaded to Postgres.')
except Exception as e:
    print('Skipping DB load (connection issue?):', e)

Schema loaded to Postgres.


## Build Generator Config
Serialize `SchemaMeta` into a JSON config that drives synthetic generation.

In [26]:
cfg_path = ARTIFACTS / 'generator_config.json'
schema_to_generator_config(schema, str(cfg_path))
print('Generator config written to', cfg_path)
print('\nPreview:')
print((cfg_path.read_text())[:600])

Generator config written to C:\Users\Ankita\Documents\Quadyster\SyntheticDataHub_SDK\artifacts\generator_config.json

Preview:
{
  "tables": {
    "Customer": {
      "primary_key": "customer_id",
      "fields": {
        "customer_id": {
          "type": "STRING",
          "required": true,
          "allowed_values": null
        },
        "first_name": {
          "type": "STRING",
          "required": true,
          "allowed_values": null
        },
        "last_name": {
          "type": "STRING",
          "required": true,
          "allowed_values": null
        },
        "date_of_birth": {
          "type": "DATE",
          "required": true,
          "allowed_values": null
        },
        "email"


## Generate Synthetic Data
Create dataframes per table, then apply relationships to assign foreign keys.

In [27]:
data = generate_synthetic_data(str(cfg_path), rows_per_table=100)
print('Tables generated:', list(data.keys()))

# Show sample rows for each table
import pandas as pd
for tname, df in data.items():
    print(f"\n## {tname} (rows={len(df)})")
    display(df.head(5))

Tables generated: ['Customer', 'Branch', 'Account', 'Card', 'Merchant', 'Transaction', 'Loan', 'LoanPayment']

## Customer (rows=100)


,customer_id,first_name,last_name,date_of_birth,email,created_at,kyc_status
0,CUS_000001,exactly,picture,2020-03-29,lay,2025-11-13 01:22:59,VERIFIED
1,CUS_000002,magazine,simply,2018-08-16,draw,2025-06-12 04:03:39,VERIFIED
2,CUS_000003,picture,risk,2017-09-25,cup,2025-02-28 21:21:44,REJECTED
3,CUS_000004,church,section,2022-11-11,test,2024-08-26 09:07:54,REJECTED
4,CUS_000005,argue,boy,2024-02-14,not,2024-06-08 16:50:55,VERIFIED



## Branch (rows=100)


,branch_id,branch_name,city,country
0,BRA_000001,alone,chance,however
1,BRA_000002,since,perform,bank
2,BRA_000003,six,computer,write
3,BRA_000004,natural,voice,nor
4,BRA_000005,shoulder,look,where



## Account (rows=100)


...

,txn_id,account_id,card_id,merchant_id,txn_timestamp,amount,currency,txn_type,status
0,TRA_000001,ACC_000043,CAR_000098,MER_000086,2024-06-01 21:51:02,18506.82,would,CREDIT,DECLINED
1,TRA_000002,ACC_000096,CAR_000024,MER_000089,2025-10-17 14:17:30,12677.63,head,DEBIT,REVERSED
2,TRA_000003,ACC_000069,CAR_000084,MER_000055,2024-07-10 00:28:37,76751.42,stuff,CREDIT,PENDING
3,TRA_000004,ACC_000095,CAR_000069,MER_000098,2025-07-24 08:37:57,15690.35,wish,CREDIT,REVERSED
4,TRA_000005,ACC_000058,CAR_000075,MER_000072,2025-04-19 02:49:08,31240.60,positive,REFUND,PENDING



## Loan (rows=100)


,loan_id,customer_id,account_id,principal_amount,interest_rate,start_date,end_date,status
0,LOA_000001,CUS_000095,ACC_000016,59151.53,3064.85,2023-05-07,2020-03-10,DEFAULTED
1,LOA_000002,CUS_000095,ACC_000054,572.71,78593.79,2024-09-26,2018-06-26,ACTIVE
2,LOA_000003,CUS_000028,ACC_000008,5263.61,24593.39,2020-05-04,2020-05-24,ACTIVE
3,LOA_000004,CUS_000035,ACC_000041,11363.74,54404.11,2016-03-21,2023-03-29,DEFAULTED
4,LOA_000005,CUS_000070,ACC_000035,97660.03,89040.29,2018-09-18,2019-04-08,CLOSED



## LoanPayment (rows=100)


,payment_id,loan_id,payment_date,amount,method
0,LOA_000001,LOA_000014,2017-11-09,17125.54,CASH
1,LOA_000002,LOA_000025,2024-03-04,13747.02,CARD
2,LOA_000003,LOA_000094,2018-08-29,15467.19,CARD
3,LOA_000004,LOA_000049,2018-04-23,85088.26,CASH
4,LOA_000005,LOA_000097,2021-06-25,59739.46,CARD


## (Optional) Write Data to Postgres
Append generated data to Postgres tables for downstream validation.

In [29]:
# Write generated data to Postgres
from db_writer import write_to_postgres
try:
    write_to_postgres(data)
    print('Data written to Postgres.')
except Exception as e:
    print('Skipping write (connection issue?):', e)

ImportError: cannot import name 'write_to_postgres' from 'db_writer' (C:\Users\Ankita\Documents\Quadyster\SyntheticDataHub_SDK\src\db_writer.py)

## (Optional) Validate PK & FK
Run PK uniqueness and FK consistency checks against the Postgres database.

In [31]:
# Validate PK & FK constraints (in-memory first, DB optional)
import importlib, validation
importlib.reload(validation)
from validation import validate_in_memory, validate_pk_fk

print('=== In-memory validation ===')
validate_in_memory(schema, data)

print('\n=== Optional DB validation ===')
try:
    validate_pk_fk(schema)
except Exception as e:
    print('Skipping DB validation (connection issue?):', e)

=== In-memory validation ===
[PK OK] Customer.customer_id
[PK OK] Branch.branch_id
[PK OK] Account.account_id
[PK OK] Card.card_id
[PK OK] Merchant.merchant_id
[PK OK] Transaction.txn_id
[PK OK] Loan.loan_id
[PK OK] LoanPayment.payment_id
[FK OK] Account.customer_id -> Customer.customer_id
[FK OK] Account.branch_id -> Branch.branch_id
[FK OK] Card.account_id -> Account.account_id
[FK OK] Transaction.account_id -> Account.account_id
[FK OK] Transaction.card_id -> Card.card_id
[FK OK] Transaction.merchant_id -> Merchant.merchant_id
[FK OK] Loan.customer_id -> Customer.customer_id
[FK OK] Loan.account_id -> Account.account_id
[FK OK] LoanPayment.loan_id -> Loan.loan_id

=== Optional DB validation ===
Skipping DB validation (connection issue?): '>' not supported between instances of 'NoneType' and 'int'


### Notes
- Adjust rows per table based on scale needs.
- Customize field generation logic by extending `synthetic_generator.py`.
- If Docker is stopped, restart with `docker compose up -d`.

In [8]:
# Debug: inspect EDL entity and attribute extraction
from pathlib import Path
import re
text = Path(EDL).read_text()
entities = re.findall(r"ENTITY\s+(\w+)\s*{", text)
print('Entity headers found:', len(entities), entities[:3])
import importlib, edl_parser
importlib.reload(edl_parser)
from edl_parser import _find_blocks
eblocks = _find_blocks(text, r"\bENTITY\s+(\w+)\s*{")
print('Balanced entity blocks:', len(eblocks))
print('First entity name:', eblocks[0][0] if eblocks else None)
print('First entity body sample:', eblocks[0][1][:200].replace('\n',' '))
attrs_first = re.findall(r"ATTRIBUTE\s+(\w+)\s*{(.*?)}", eblocks[0][1], re.S) if eblocks else []
print('Attributes in first entity:', len(attrs_first), [a[0] for a in attrs_first][:5])

Entity headers found: 8 ['Customer', 'Branch', 'Account']
Balanced entity blocks: 8
First entity name: Customer
First entity body sample:      DESCRIPTION = "Customer master information"      ATTRIBUTE customer_id {         TYPE = STRING         LENGTH = 36         REQUIRED = TRUE     }      ATTRIBUTE first_name {         TYPE = STRING 
Attributes in first entity: 7 ['customer_id', 'first_name', 'last_name', 'date_of_birth', 'email']


In [19]:
# Debug import of db_writer
import importlib, traceback
try:
    import db_writer
    importlib.reload(db_writer)
    print('db_writer loaded. Symbols:', [s for s in dir(db_writer) if not s.startswith('_')])
except Exception as e:
    print('db_writer import failed:', e)
    traceback.print_exc()

db_writer loaded. Symbols: []


In [20]:
# Inspect db_writer dict
import importlib, db_writer
importlib.reload(db_writer)
print(list(db_writer.__dict__.keys())[:20])
print('Has write_to_postgres:', hasattr(db_writer, 'write_to_postgres'))
print('Module file:', getattr(db_writer, '__file__', None))

['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__file__', '__cached__', '__builtins__']
Has write_to_postgres: False
Module file: C:\Users\Ankita\Documents\Quadyster\SyntheticDataHub_SDK\src\db_writer.py


In [ ]:
# Show db_writer source as seen by the kernel
from pathlib import Path
print((SRC / 'db_writer.py').read_text())

In [ ]:
# Force-reload db_writer and import function
import sys, importlib
if 'db_writer' in sys.modules:
    del sys.modules['db_writer']
import db_writer
importlib.reload(db_writer)
print('Has write_to_postgres:', hasattr(db_writer, 'write_to_postgres'))
from db_writer import write_to_postgres
print('Imported write_to_postgres successfully')